# Flux — scFEA metabolni flux (TRIM-Flux Var 2)

Izracuna metabolni flux za vsako celico prek **scFEA** (single-cell Flux Estimation Analysis).
Flux postane 3. modaliteta v TRIM (poleg RNA + TCR).

**Vhod:** `data_rna_counts.pkl` (SUROVI counti — scFEA jih sam normalizira; NE normalizirani `data_rna.pkl`)
**Izhod:** `data_flux.pkl` — matrika (celice x ~168 metabolnih modulov), poravnana z `data_labels`

Orodje scFEA (izbrano po raziskavi izvedljivosti): GNN, GPU, per-cell, dropout-robusten (korelacija >0.85),
~168 cloveskih metabolnih modulov. Nevzdrzevan od 2021 -> potrebni patchi za moderni Colab (spodaj).

> scFEA fluksi so RELATIVNI/model-odvisni (ne absolutne hitrosti); benchmark scFEA/Compass/METAFlux ne obstaja.


## 0. Mount + namestitev scFEA (+ patchi za moderni Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -q https://github.com/changwn/scFEA.git
%cd /content/scFEA

# scFEA importa 'magic' na vrhu skripte -> nujno nalozen (tudi pri sc_imputation=False).
# --no-deps: sicer vlece star pandas iz vira -> build pade.
!pip install -q --no-deps magic-impute graphtools scprep s_gd2 pygsp Deprecated tasklogger wrapt

# PATCHI (scFEA pisan za pandas<2 / stari torch):
# pandas 3: .append IN ._append odstranjena -> pd.concat (dela na pandas 2 in 3)
# Ciljaj TOCNO 'geneExprDf.append(temp,' (DataFrame), NE list.append.
!sed -i 's/geneExprDf = geneExprDf\.append(temp,/geneExprDf = pd.concat([geneExprDf, temp],/' src/scFEA.py
# pandas 3: concat iz praznega DataFrame da object dtype -> torch pade. Vsili float.
!sed -i 's/X = geneExprDf\.values\.T/X = geneExprDf.values.T.astype("float32")/' src/scFEA.py
!sed -i 's/module_scale = df\.groupby(df\.index)\.sum()\.T/module_scale = df.groupby(df.index).sum().T.astype("float64")/' src/scFEA.py
!sed -i 's/\.detach()\.numpy()/.detach().cpu().numpy()/g' src/scFEA.py  # torch: GPU tensor -> cpu

import magic
print('magic:', getattr(magic, '__version__', 'OK'))
print('cmMat na voljo:')
!ls data/ | grep -iE 'module_gene|cmMat'

## 1. Nalozi surove counte + gene imena

scFEA hoce **surove counte** (sam logira ce max>50) in **gene simbole** (vrstice=geni, stolpci=celice).
`data_rna_counts.pkl` je shranjen v notebooku 01 (loceno od normaliziranega `data_rna.pkl`).

In [ ]:
import pickle, numpy as np, pandas as pd, os, time
from scipy.sparse import issparse

DATA = '/content/drive/MyDrive/Diploma/data/processed'
with open(os.path.join(DATA, 'data_rna_counts.pkl'), 'rb') as f:
    counts = pickle.load(f)                 # SUROVI counti (celice x geni), sparse
with open(os.path.join(DATA, 'gene_names.pkl'), 'rb') as f:
    gene_names = [str(g) for g in pickle.load(f)]

print('counts:', counts.shape, type(counts).__name__)
print('gene_names:', len(gene_names), '| primer:', gene_names[:4])
assert len(gene_names) == counts.shape[1], 'gene_names != stolpci counts!'

# scFEA rabi gene SIMBOLE (CD8A), ne Ensembl ID (ENSG...).
is_ensembl = all(g.upper().startswith('ENSG') for g in gene_names[:50])
assert not is_ensembl, 'Geni so Ensembl ID -> scFEA rabi simbole (pretvori z mygene)!'
print('Geni so simboli:', not is_ensembl)

In [ ]:
# === DIAGNOSTIKA GPU (pred dolgim zagonom!) ===
# Colab kaze le 0.4 GB GPU RAM -> preveri ce scFEA sploh uporablja GPU.
# scFEA (scFEA.py:94): device = cuda ce torch.cuda.is_available() SICER cpu (tiho!).
import torch
print('torch:', torch.__version__)
print('CUDA na voljo (torch.cuda.is_available()):', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    # hiter GPU test: majhen matmul + koliko RAM zasede
    x = torch.randn(5000, 658, device='cuda')
    w = torch.randn(658, 168, device='cuda')
    _ = (x @ w).sum().item()
    print(f'GPU test OK | zaseden GPU RAM: {torch.cuda.memory_allocated()/1e9:.3f} GB '
          f'(majhen model -> malo RAM je NORMALNO, ne dokaz da GPU ne dela)')
    print('-> scFEA BO uporabil GPU (is_available=True). 0.4 GB je OK za tako majhen GNN.')
    del x, w; torch.cuda.empty_cache()
else:
    print('!!! CUDA NI na voljo -> scFEA bo tekel na CPU (10x+ pocasneje = 2h/batch).')
    print('!!! USTAVI: Runtime > Change runtime type > Hardware accelerator > GPU (T4/A100).')
# preveri tudi Colab nvidia-smi
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv 2>/dev/null || echo 'nvidia-smi ni na voljo -> verjetno CPU runtime'

## 2. Batch scFEA -> flux (RESUMABLE, za nocni zagon)

70k celic naenkrat crasha RAM (scFEA "process data" zanka = 168 kopij matrike),
zato kosi po CHUNK celic. scFEA trenira GNN 100 epoch/batch -> DOLGO (~ure/batch).

**RESUMABLE:** vsak batch se takoj shrani na **Drive** (ne lokalni disk!). Ob resetu
runtime (Colab pogosto odklopi cez noc) samo znova zazeni celico -> preskoci ze
narejene batche, nadaljuje kjer je ostal. NIC se ne izgubi.

Prvi batch IZPISE device (cuda vs cpu). Ce "cpu" -> trening bo 10x pocasnejsi ->
ustavi, preklopi na GPU runtime (Runtime > Change runtime type > GPU).

Izhod: `data_flux.pkl` (celice x ~168 modulov), poravnan z `data_labels`.

In [ ]:
# ============================================================================
# BATCH scFEA -> flux (RESUMABLE za nocni zagon)
#
# scFEA trenira GNN 100 epoch na CEL batch naenkrat (BATCH_SIZE = vse celice) ->
# ~ure na batch. 70k celic naenkrat = RAM crash ("process data" 168x kopira matriko).
# Resitev: kosi po CHUNK, vsak scFEA locen, ZAPOREDNO (scFEA zasede cel GPU).
#
# RESUMABLE: vsak batch flux CSV -> na DRIVE (ne /content/, ki izgine ob resetu).
# CE CRASHA: samo znova zazeni to celico -> izpise "RESUME: [0,1,..]" -> preskoci
# narejene -> nadaljuje kjer je ostal. NIC ne brises, NIC rocno.
# ============================================================================
from tqdm import tqdm
import json

# --- resumable mapa NA DRIVE (prezivi reset runtime) ---
FLUX_DIR = os.path.join(DATA, 'flux_batches')     # /content/drive/.../flux_batches/
os.makedirs(FLUX_DIR, exist_ok=True)
print('Batch izhodi (Drive, resumable):', FLUX_DIR)

# 1) filtriraj na scFEA modulne gene (55x manjsi vhod, identicen rezultat)
mg = pd.read_csv('/content/scFEA/data/module_gene_m168.csv', index_col=0)
module_genes = set()
for col in mg.columns:
    for v in mg[col].dropna().astype(str):
        g = v.strip()
        if g and g.lower() != 'nan': module_genes.add(g)
keep_idx = [i for i, g in enumerate(gene_names) if g in module_genes]
keep_names = [gene_names[i] for i in keep_idx]
print(f'scFEA modulnih genov: {len(module_genes)} | nasih v modulih: {len(keep_idx)}')
assert len(keep_idx) > 100, 'premalo ujemanja gene-imen!'

counts_k = counts[:, keep_idx]                 # sparse, le modulni geni
N = counts_k.shape[0]
# CHUNK: skupni cas ~fiksen (70k celic x 100 epoch), a manjsi kos = ob crashu izgubis
# manj (le zadnji nedokoncani batch). 5000 -> 14 batchov; 10000 -> 7. Oba OK za RAM.
CHUNK = 5000
n_batches = (N + CHUNK - 1) // CHUNK

# --- VAROVALKA: CHUNK se NE sme spremeniti med resume (sicer c{i} indeksi razni!) ---
# ob prvem zagonu shrani (N, CHUNK); ob resume preveri ujemanje.
meta_path = os.path.join(FLUX_DIR, 'run_meta.json')
if os.path.exists(meta_path):
    meta = json.load(open(meta_path))
    assert meta['N'] == N and meta['CHUNK'] == CHUNK, (
        f"NEUJEMANJE! Zacel si z N={meta['N']}, CHUNK={meta['CHUNK']}, zdaj N={N}, CHUNK={CHUNK}.\n"
        f"Batchi na Drive so poravnani na STARI CHUNK. Ali vrni CHUNK={meta['CHUNK']}, "
        f"ali izbrisi mapo {FLUX_DIR} in zacni znova.")
else:
    json.dump({'N': int(N), 'CHUNK': int(CHUNK)}, open(meta_path, 'w'))
print(f'Celic: {N} | kosov po {CHUNK}: {n_batches}')

os.makedirs('/content/scfea_input', exist_ok=True)
%cd /content/scFEA

# --- koliko je ze narejenih (resume) ---
done = [b for b in range(n_batches) if os.path.exists(os.path.join(FLUX_DIR, f'flux_{b}.csv'))]
if done:
    print(f'RESUME: ze narejeni batchi {done} ({len(done)}/{n_batches}) -> preskocim jih.')

printed_device = False                          # device izpis samo enkrat (prvi izvedeni batch)
t0 = time.time()
for b in tqdm(range(n_batches), desc='scFEA batchi'):
    out_csv = os.path.join(FLUX_DIR, f'flux_{b}.csv')
    if os.path.exists(out_csv):
        continue                                # ze narejen (prezivel reset) -> preskoci

    lo, hi = b*CHUNK, min((b+1)*CHUNK, N)
    Xk = counts_k[lo:hi]
    Xk = Xk.toarray() if issparse(Xk) else np.asarray(Xk)
    Xk = np.rint(Xk).astype(np.int32)
    # CSV: geni x celice; imena celic = GLOBALNI indeks (c{lo}..) -> unikatna cez kose
    df_in = pd.DataFrame(Xk.T, index=keep_names, columns=[f'c{i}' for i in range(lo, hi)])
    df_in.to_csv('/content/scfea_input/expr.csv')
    del Xk, df_in

    # scFEA pise na LOKALNI disk (hitro), potem kopiramo na Drive (atomarno)
    tmp_csv = f'/content/flux_{b}.csv'
    log_txt = os.path.join(FLUX_DIR, f'log_{b}.txt')      # log NA Drive (za diagnostiko)
    rc = os.system(
        f'python src/scFEA.py --data_dir data --input_dir /content/scfea_input '
        f'--test_file expr.csv --moduleGene_file module_gene_m168.csv '
        f'--stoichiometry_matrix cmMat_c70_m168.csv '
        f'--output_flux_file {tmp_csv} --output_balance_file /content/bal_{b}.csv '
        f'--sc_imputation False > "{log_txt}" 2>&1')

    if rc != 0 or not os.path.exists(tmp_csv):
        print(f'\n!!! batch {b} PADEL -- zadnjih 20 vrstic log:')
        os.system(f'tail -n 20 "{log_txt}"')
        raise RuntimeError(f'batch {b} padel (log na Drive: {log_txt})')

    # DIAGNOSTIKA prvega izvedenega batcha: preveri torch device (GPU vs CPU)
    if not printed_device:
        printed_device = True
        print(f'\n  [torch device: {"cuda GPU" if torch.cuda.is_available() else "CPU (!) -> 10x pocasneje, preklopi na GPU runtime"}]')

    # preberi + preveri + shrani na Drive (samo ce je CEL in brez NaN)
    fb = pd.read_csv(tmp_csv, index_col=0)
    assert fb.shape[0] == (hi - lo), f'batch {b}: {fb.shape[0]} vrstic != {hi-lo} celic!'
    assert int(np.isnan(fb.values).sum()) == 0, f'batch {b} vsebuje NaN!'
    fb.to_csv(out_csv)                          # -> na Drive (atomarno zadnji korak)
    os.remove(tmp_csv)
    print(f'  batch {b}: {fb.shape} shranjen na Drive ({time.time()-t0:.0f}s skupaj)')

print(f'\n=== vsi batchi koncani: {time.time()-t0:.0f} s ===')

# 2) zdruzi VSE batch CSV-je z DRIVE (deluje tudi po resetu: samo poZeni celico)
flux_parts = []
for b in range(n_batches):
    fp = os.path.join(FLUX_DIR, f'flux_{b}.csv')
    assert os.path.exists(fp), f'MANJKA batch {b} ({fp})! Znova zaZeni celico.'
    flux_parts.append(pd.read_csv(fp, index_col=0))
flux = pd.concat(flux_parts, axis=0)
print('Zdruzen flux:', flux.shape)

# 3) poravnaj na NAS vrstni red celic (c0..cN-1) + preveri
expected = [f'c{i}' for i in range(N)]
assert set(expected) == set(flux.index.astype(str)), 'flux celice se ne ujemajo!'
flux = flux.loc[expected]

n_nan = int(np.isnan(flux.values).sum())
print('NaN:', n_nan, '| delez nicelnih:', f'{(flux.values==0).mean():.3f}',
      '| min/max/mean:', f'{flux.values.min():.3f}/{flux.values.max():.3f}/{flux.values.mean():.3f}')
assert n_nan == 0, 'FLUX VSEBUJE NaN!'
assert flux.shape[0] == counts.shape[0], f'flux vrstic ({flux.shape[0]}) != celic ({counts.shape[0]})!'

# 4) shrani koncni data_flux.pkl (poravnan z data_labels po vrsticah, kot data_rna/data_tcr)
data_flux = flux.values.astype(np.float32)
with open(os.path.join(DATA, 'data_flux.pkl'), 'wb') as f:
    pickle.dump(data_flux, f)
with open(os.path.join(DATA, 'flux_module_names.pkl'), 'wb') as f:
    pickle.dump(list(flux.columns), f)
print(f'\ndata_flux.pkl shranjen: {data_flux.shape} (3. modaliteta za TRIM)')

## 5. Zakljucek

- `data_flux.pkl` (celice x ~168 modulov) = metabolni flux, poravnan z RNA/TCR.
- Naslednje: flux encoder/decoder kot 3. modaliteta v TRIM (RNA + TCR + Flux, Var 2).

Pridrzki za diplomo: scFEA fluksi RELATIVNI/model-odvisni; scFEA nevzdrzevan (patchi za pandas/torch).